In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/dados_processados.csv')
print(f"Total de registros: {df.shape[0]}")
df.tail()


In [0]:
df.describe()


In [0]:
import seaborn as sns
import matplotlib.pyplot as plt


labels_map = {0: 'Sem Alagamento', 1: 'Com Alagamento'}
counts = (df['historico_alagamento']
          .map(labels_map)
          .value_counts()
          .reindex(['Sem Alagamento', 'Com Alagamento'], fill_value=0))

def autopct_fmt(values):
    total = sum(values)
    def _inner(pct):
        val = int(round(pct * total / 100.0))
        return f'{pct:.1f}%\n({val})' 
    return _inner

ax = counts.plot(
    kind='pie',
    autopct=autopct_fmt(counts.values),
    labels=counts.index,
    startangle=90,
    ylabel='' 
)

plt.title("Distribuição das classes")
plt.tight_layout()
plt.show()


In [0]:


df_alagou = df[df['historico_alagamento'] == 1]
df_nao_alagou = df[df['historico_alagamento'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_alagou['precipitacao_diaria'], bins=10, kde=True, color='blue', ax=axes[0])
axes[0].set_title('Precipitação - Alagamento')
axes[0].set_xlabel('Precipitação (mm)')
axes[0].set_ylabel('Frequência')
axes[0].grid(True)

sns.histplot(df_nao_alagou['precipitacao_diaria'], bins=10, kde=True, color='green', ax=axes[1])
axes[1].set_title('Precipitação - Sem Alagamento')
axes[1].set_xlabel('Precipitação (mm)', )
axes[1].set_ylabel('Frequência')
axes[1].set_xlim(left=0) 
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../reports/histograma_precipitacao_alagamento.png', dpi=300, bbox_inches='tight')
plt.show()


In [0]:
sns.boxplot(x='historico_alagamento', y='precipitacao_chuva', data=df)
plt.savefig('../reports/boxplot_precipitacao_alagamento.png', dpi=300, bbox_inches='tight')


In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['historico_alagamento'] == 1]['intensidade_chuva']\
    .value_counts().head(5).plot(kind='bar', ax=axes[0], color='blue')
axes[0].set_title('Top 5 Intensidade (Alagou)')
axes[0].set_ylabel('Frequência')

df[df['historico_alagamento'] == 0]['intensidade_chuva']\
    .value_counts().head(5).plot(kind='bar', ax=axes[1], color='green')
axes[1].set_title('Top 5 Intensidade (Não Alagou)')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.savefig('../reports/topintensidade_chuva.png', dpi=300, bbox_inches='tight')
plt.show()


In [0]:


plt.figure(figsize=(12, 8))  
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Mapa de Correlação entre Variáveis Numéricas')
plt.tight_layout()
plt.savefig('../reports/mapa_correlacao_variaveis.png', dpi=300, bbox_inches='tight')

plt.show()


In [0]:


sns.kdeplot(df[df['historico_alagamento']==1]['pressao'], label='Alagamento')
sns.kdeplot(df[df['historico_alagamento']==0]['pressao'], label='Sem Alagamento')
plt.legend()
plt.title("Distribuição da pressão atmosférica")
plt.savefig('../reports/distribuico_pressao_atmosferica.png', dpi=300, bbox_inches='tight')



In [0]:
import seaborn as sns
sns.boxplot(x='historico_alagamento', y='tempo_chuva', data=df)
plt.savefig('../reports/boxplot_tempochuva.png', dpi=300, bbox_inches='tight')


In [0]:
pos = df[df['historico_alagamento'] == 1].copy()

plt.figure(figsize=(6,4))
sns.histplot(pos['declive_graus'], bins=40, kde=True)
plt.title('Distribuição do Declive — apenas casos com alagamento')
plt.xlabel('Declive (°)'); plt.ylabel('Contagem')
plt.tight_layout(); plt.show()

print('mediana(declive) alagou =', pos['declive_graus'].median())


In [0]:
plt.figure(figsize=(7,4))
sns.kdeplot(data=df, x='declive_graus', hue='historico_alagamento',
            common_norm=False, fill=True, alpha=0.25)
plt.title('Declive — KDE por classe (0/1)')
plt.xlabel('Declive (°)'); plt.tight_layout(); plt.show()

print('mediana 0 =', df.loc[df.historico_alagamento==0, 'declive_graus'].median())
print('mediana 1 =', df.loc[df.historico_alagamento==1, 'declive_graus'].median())


In [0]:
bins = [0,2,4,6,8,10,15,20,30,60]
labels = [f'{bins[i]}–{bins[i+1]}°' for i in range(len(bins)-1)]
pos = df[df['historico_alagamento']==1].copy()
pos['declive_bin'] = pd.cut(pos['declive_graus'], bins=bins, labels=labels,
                            include_lowest=True, right=False)

comp = (pos['declive_bin']
          .value_counts(normalize=True)
          .sort_index()
          .mul(100)
          .reset_index(name='percentual')   
          .rename(columns={'index':'declive_bin'}))


plt.figure(figsize=(9,4))
ax = sns.barplot(data=comp, x='declive_bin', y='percentual')
for p, v in zip(ax.patches, comp['percentual']):
    ax.annotate(f'{v:.1f}%', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9, xytext=(0,3), textcoords='offset points')
plt.title('Composição dos casos com alagamento por faixa de declive')
plt.xlabel('Faixa de Declive (°)'); plt.ylabel('% dentro dos alagamentos')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()
